# 🚀 4-Hour Hands-On Tutorial: LangChain with Google Gemini

**Platform:** Google Colab  
**Language:** Python  
**LLM:** Google Gemini through `langchain-google-genai`  
**Primary model:** `gemini-3.5-flash`  
**Level:** Beginner → Intermediate  
**Duration:** Approximately 4 hours

---

## What you will build

By the end of this notebook, you will have hands-on experience with:

1. Connecting LangChain to Google Gemini.
2. Sending prompts to Gemini through LangChain.
3. Using prompt templates.
4. Building chains with LCEL (`|` operator).
5. Parsing text and structured outputs.
6. Building a simple conversational assistant.
7. Creating custom tools.
8. Building a Retrieval-Augmented Generation (RAG) pipeline.
9. Completing a mini-project: **AI Customer Support Knowledge Assistant**.

> **Important:** Never hard-code your API key into notebooks that you intend to share publicly.


## ⏱️ Suggested 4-Hour Agenda

| Time | Module | Hands-on Outcome |
|---|---|---|
| 0:00–0:25 | Module 1 — Setup and Gemini basics | Connect Gemini to LangChain |
| 0:25–1:05 | Module 2 — Prompt templates and chains | Build reusable LCEL chains |
| 1:05–1:35 | Module 3 — Output parsing and structured data | Extract validated business data |
| 1:35–2:05 | Module 4 — Conversation and message history | Build a context-aware chatbot |
| 2:05–2:35 | Module 5 — Tools and tool calling | Give the model callable functions |
| 2:35–3:20 | Module 6 — Embeddings and RAG | Ask questions over private documents |
| 3:20–4:00 | Module 7 — Mini-project | Build a customer-support knowledge assistant |


# Module 1 — Setup and Your First Gemini Call

## 1.1 What is LangChain?

LangChain is an application framework for building AI systems that combine language models with prompts, tools, retrieval systems, structured outputs, and application logic.

A useful mental model is:

**User Question → Prompt → Model → Optional Tools/Retrieval → Structured Response**

In this tutorial, Google Gemini is the language model, while LangChain is the orchestration layer.


## 1.2 Install the required packages

Run the following cell in Google Colab.

The notebook installs:

- `langchain`
- `langchain-google-genai`
- `langchain-community`
- `langchain-text-splitters`
- `pydantic`

After installation, Colab may occasionally ask you to restart the runtime when dependency versions change.


In [ ]:
%pip install -qU langchain langchain-google-genai langchain-community langchain-text-splitters pydantic

## 1.3 Add your Google Gemini API key securely

Create a Gemini API key from Google AI Studio, then enter it below.

The input is hidden by `getpass`, so the key is not displayed in the notebook output.


In [ ]:
import os
import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("✅ GOOGLE_API_KEY is configured for this Colab session.")

## 1.4 Initialize the Gemini chat model

For this tutorial we use `gemini-3.5-flash`.

`temperature` controls response randomness. A higher value tends to produce more varied answers.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    max_retries=2,
)

print("✅ Gemini model initialized.")

## 1.5 Your first model call

In [ ]:
response = llm.invoke("Explain LangChain in simple terms for a beginner in three sentences.")
print(response.content)

### Exercise 1

Change the prompt and ask Gemini to explain one of these topics in simple terms:

- RAG
- AI agents
- Vector databases
- Prompt engineering

Try adding constraints such as: *Use an analogy* or *Explain in exactly five bullet points*.


In [ ]:
# TODO: Replace the prompt with your own question.

exercise_response = llm.invoke(
    "Explain Retrieval-Augmented Generation using a library analogy."
)
print(exercise_response.content)

# Module 2 — Prompt Templates and LCEL Chains

## 2.1 Why use prompt templates?

Hard-coded prompts become difficult to maintain when an application has many users or changing inputs.

A prompt template separates:

- fixed instructions;
- dynamic variables;
- reusable application logic.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher. Explain concepts clearly for a beginner."),
    ("human", "Explain {topic} using a real-world example.")
])

formatted_prompt = prompt.invoke({"topic": "vector embeddings"})
print(formatted_prompt)

## 2.2 Build your first LCEL chain

LangChain Expression Language, commonly called **LCEL**, lets us connect components with the pipe operator:

`prompt | model | parser`


In [ ]:
from langchain_core.output_parsers import StrOutputParser

text_parser = StrOutputParser()

basic_chain = prompt | llm | text_parser

result = basic_chain.invoke({"topic": "AI agents"})
print(result)

## 2.3 Build a business-oriented chain

Imagine that a manager wants a concise explanation of an AI use case.


In [ ]:
business_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a business AI consultant. Return a concise and practical explanation."
    ),
    (
        "human",
        """Analyze the following AI use case:

Use case: {use_case}
Industry: {industry}

Explain:
1. Business problem
2. How AI can help
3. Required data
4. Expected business value
5. One implementation risk
"""
    )
])

business_chain = business_prompt | llm | StrOutputParser()

business_result = business_chain.invoke({
    "use_case": "Predict which customers are likely to churn",
    "industry": "Telecommunications"
})

print(business_result)

## 2.4 Chain composition

A major LangChain idea is that the output of one component can become the input of another.

Below, the first chain creates a summary and the second chain creates quiz questions from that summary.


In [ ]:
summary_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} for a beginner in approximately 150 words."
)

quiz_prompt = ChatPromptTemplate.from_template(
    """Based only on the following explanation, create five multiple-choice questions.
Each question must have four options and provide the correct answer.

Explanation:
{explanation}
"""
)

summary_chain = summary_prompt | llm | StrOutputParser()
quiz_chain = quiz_prompt | llm | StrOutputParser()

explanation = summary_chain.invoke({"topic": "supervised machine learning"})
quiz = quiz_chain.invoke({"explanation": explanation})

print("=== EXPLANATION ===")
print(explanation)

print("\n=== QUIZ ===")
print(quiz)

### Exercise 2

Build a chain that accepts:

- a job role;
- a technical topic;
- the learner's experience level.

The chain should produce a personalized 7-day learning plan.

A starter solution is provided below. Modify it and experiment.


In [ ]:
learning_prompt = ChatPromptTemplate.from_template(
    """You are an expert technical trainer.

Create a 7-day learning plan.

Job role: {role}
Topic: {topic}
Experience level: {level}

For each day provide:
- Learning objective
- Topics
- One hands-on task
- Expected outcome

Keep the plan practical.
"""
)

learning_chain = learning_prompt | llm | StrOutputParser()

plan = learning_chain.invoke({
    "role": "SAP Developer",
    "topic": "Generative AI and LangChain",
    "level": "Beginner"
})

print(plan)

# Module 3 — Structured Output

Free-form text is useful for humans, but applications often need predictable fields.

For example, a support application may need:

```text
category
priority
sentiment
summary
recommended_team
```

LangChain can ask supported models to return output matching a Pydantic schema.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    category: Literal[
        "Billing",
        "Technical",
        "Account",
        "Delivery",
        "Other"
    ] = Field(description="Primary ticket category")

    priority: Literal["High", "Medium", "Low"] = Field(
        description="Urgency of the issue"
    )

    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(
        description="Customer sentiment"
    )

    summary: str = Field(
        description="Short summary of the customer problem"
    )

    recommended_team: str = Field(
        description="Team that should handle the issue"
    )

structured_llm = llm.with_structured_output(SupportTicket)

ticket_text = """
I have been charged twice for my subscription this month.
I contacted support yesterday but have not received a reply.
Please refund the duplicate charge immediately.
"""

ticket = structured_llm.invoke(
    f"Analyze this customer support ticket:\n\n{ticket_text}"
)

ticket

## 3.1 Convert the validated object to a dictionary

In [ ]:
ticket.model_dump()

### Exercise 3 — Employee feedback analyzer

Create a schema containing:

- `main_topic`
- `sentiment`
- `urgency`
- `summary`
- `action_required`

Then analyze the feedback below.


In [ ]:
class EmployeeFeedback(BaseModel):
    main_topic: str
    sentiment: Literal["Positive", "Neutral", "Negative"]
    urgency: Literal["High", "Medium", "Low"]
    summary: str
    action_required: bool

feedback_analyzer = llm.with_structured_output(EmployeeFeedback)

feedback = """
The new internal learning portal is useful, but search is extremely slow.
I need to complete mandatory training by tomorrow and the portal repeatedly times out.
"""

feedback_result = feedback_analyzer.invoke(
    f"Analyze the following employee feedback:\n\n{feedback}"
)

feedback_result

# Module 4 — Conversation and Message History

A basic model call does not automatically remember previous Python calls.

To build a conversation, we need to pass prior messages back to the model.

LangChain provides standard message classes such as:

- `HumanMessage`
- `AIMessage`
- `SystemMessage`


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

messages = [
    SystemMessage(
        content="You are a helpful AI tutor. Keep answers concise and beginner-friendly."
    ),
    HumanMessage(content="My name is Prem and I am learning LangChain."),
]

first_reply = llm.invoke(messages)
print(first_reply.content)

messages.append(first_reply)
messages.append(HumanMessage(content="What is my name and what am I learning?"))

second_reply = llm.invoke(messages)
print("\n" + second_reply.content)

## 4.1 Build a simple conversational loop

Run the following cell and chat until you type `exit`.

> In a production system, conversation history would usually be persisted in a database or managed through a dedicated state layer rather than kept only in a Python list.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

chat_history = [
    SystemMessage(
        content="You are a patient LangChain tutor. Give concise, practical answers."
    )
]

print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    chat_history.append(HumanMessage(content=user_input))

    assistant_response = llm.invoke(chat_history)
    chat_history.append(assistant_response)

    print(f"Gemini: {assistant_response.content}\n")

# Module 5 — Tools and Tool Calling

A **tool** is a function that an AI model can request to use.

Examples:

- get weather;
- query a database;
- calculate loan payments;
- search product inventory;
- create a support ticket.

We will first create simple local tools.


In [ ]:
from langchain_core.tools import tool

@tool
def calculate_discount(price: float, discount_percent: float) -> float:
    """Calculate the final price after applying a percentage discount."""
    return round(price * (1 - discount_percent / 100), 2)

@tool
def get_order_status(order_id: str) -> str:
    """Get the status of an order from a simulated order database."""
    mock_orders = {
        "ORD-1001": "Shipped",
        "ORD-1002": "Processing",
        "ORD-1003": "Delivered",
    }
    return mock_orders.get(order_id, "Order not found")

print(calculate_discount.invoke({"price": 1000, "discount_percent": 15}))
print(get_order_status.invoke({"order_id": "ORD-1001"}))

## 5.1 Bind tools to Gemini

Binding tools tells the model what functions are available.

The model may return a tool-call request instead of final natural-language text.


In [ ]:
tools = [calculate_discount, get_order_status]
llm_with_tools = llm.bind_tools(tools)

tool_response = llm_with_tools.invoke(
    "What is the status of order ORD-1002?"
)

print("Content:", tool_response.content)
print("Tool calls:", tool_response.tool_calls)

## 5.2 Execute requested tool calls manually

This demonstrates the essential mechanics behind tool-using agents.


In [ ]:
tool_map = {tool.name: tool for tool in tools}

user_question = "What is the final price of a ₹2500 product after a 12% discount?"
ai_message = llm_with_tools.invoke(user_question)

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        selected_tool = tool_map[tool_call["name"]]
        tool_result = selected_tool.invoke(tool_call["args"])

        print("Tool selected:", tool_call["name"])
        print("Arguments:", tool_call["args"])
        print("Tool result:", tool_result)
else:
    print(ai_message.content)

### Exercise 4 — Create your own business tool

Create a tool called `calculate_simple_interest` that accepts:

- principal;
- annual interest rate;
- number of years.

Formula:

`Simple Interest = Principal × Rate × Time / 100`

Then bind it to Gemini and ask a natural-language question.


In [ ]:
@tool
def calculate_simple_interest(
    principal: float,
    annual_rate: float,
    years: float
) -> float:
    """Calculate simple interest from principal, annual rate percentage, and time in years."""
    return round(principal * annual_rate * years / 100, 2)

finance_llm = llm.bind_tools([calculate_simple_interest])

finance_response = finance_llm.invoke(
    "Calculate the simple interest on ₹50,000 at 7.5% per year for 3 years."
)

print(finance_response.tool_calls)

# Module 6 — Embeddings and RAG

## 6.1 What is RAG?

Retrieval-Augmented Generation gives an LLM relevant information from an external knowledge source before it answers.

Typical flow:

**Documents → Split into chunks → Embeddings → Vector store → Retrieve relevant chunks → Gemini generates grounded answer**

This is useful when the model needs access to:

- company policies;
- product documentation;
- contracts;
- manuals;
- SAP documentation;
- internal knowledge bases.


## 6.2 Create sample private documents

For a real application, these could come from PDF files, websites, databases, SharePoint, SAP systems, or other repositories.

For this hands-on tutorial, we use three policy documents directly in Python.


In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
        Refund Policy:
        Customers may request a full refund within 30 calendar days of purchase.
        After 30 days, refunds are not normally available unless the product is defective.
        Approved refunds are processed within 5 to 7 business days.
        """,
        metadata={"source": "refund_policy"}
    ),
    Document(
        page_content="""
        Shipping Policy:
        Standard delivery takes 3 to 5 business days.
        Express delivery takes 1 to 2 business days.
        Orders above ₹2,000 qualify for free standard delivery.
        """,
        metadata={"source": "shipping_policy"}
    ),
    Document(
        page_content="""
        Premium Support Policy:
        Premium customers receive 24/7 email and chat support.
        Standard customers receive support Monday to Friday from 9 AM to 6 PM.
        Critical premium incidents have a target initial response time of one hour.
        """,
        metadata={"source": "support_policy"}
    ),
]

print(f"Created {len(documents)} documents.")

## 6.3 Split documents into chunks

Smaller chunks improve retrieval granularity.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content.strip())

## 6.4 Create Gemini embeddings and an in-memory vector store

This notebook uses an in-memory vector store to keep setup simple.

For larger production applications, you may use a persistent vector database.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768,
)

vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Vector store created.")

## 6.5 Test retrieval before generation

Good RAG systems inspect retrieval quality separately from final answer quality.


In [ ]:
question = "How long does a refund take after approval?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Retrieved document {i} ---")
    print(doc.page_content.strip())
    print("Source:", doc.metadata.get("source"))
    print()

## 6.6 Build the RAG chain

The model is explicitly instructed to answer only from retrieved context and admit when the context is insufficient.


In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are a customer-support knowledge assistant.

Answer the user's question using only the context below.

Rules:
1. Do not invent information.
2. If the answer is not present in the context, say:
   "I do not have enough information in the provided knowledge base."
3. Keep the answer concise.
4. Mention the relevant policy when possible.

Context:
{context}

Question:
{question}
"""
)

def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in docs
    )

def ask_rag(question: str) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    return (rag_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question
    })

print(ask_rag("Can I get a refund 20 days after purchase?"))

## 6.7 Test multiple RAG questions

In [ ]:
questions = [
    "How many days does standard shipping take?",
    "Who receives 24/7 support?",
    "What happens if I ask for a refund after 45 days?",
    "What is the company's office address?"
]

for q in questions:
    print(f"QUESTION: {q}")
    print("ANSWER:", ask_rag(q))
    print("-" * 80)

# Module 7 — Mini-Project: AI Customer Support Knowledge Assistant

## Problem statement

Build an AI assistant that:

1. receives a customer question;
2. retrieves relevant company-policy information;
3. generates a grounded answer;
4. classifies the request;
5. returns a structured result that another application could consume.

The final output will contain:

- `answer`
- `category`
- `priority`
- `needs_human_agent`
- `source_policies`


In [ ]:
from typing import List

class CustomerSupportResponse(BaseModel):
    answer: str = Field(description="Grounded answer to the customer")
    category: Literal["Refund", "Shipping", "Support", "Other"]
    priority: Literal["High", "Medium", "Low"]
    needs_human_agent: bool
    source_policies: List[str] = Field(
        description="Knowledge-base sources used for the response"
    )

support_structured_llm = llm.with_structured_output(CustomerSupportResponse)

## 7.1 Create the final application function

In [ ]:
final_support_prompt = ChatPromptTemplate.from_template(
    """You are an AI customer support assistant.

Use only the supplied knowledge-base context.

Customer question:
{question}

Knowledge-base context:
{context}

Instructions:
- Provide a concise grounded answer.
- Never invent a policy.
- Classify the request.
- Set needs_human_agent to true when:
  - the knowledge base is insufficient,
  - the request requires an exception,
  - there is an unresolved urgent complaint.
- Include only sources that appear in the supplied context.
"""
)

def customer_support_assistant(question: str) -> CustomerSupportResponse:
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)

    chain = final_support_prompt | support_structured_llm

    return chain.invoke({
        "question": question,
        "context": context
    })

## 7.2 Test the complete assistant

In [ ]:
test_questions = [
    "I bought the product 15 days ago. Can I return it for a full refund?",
    "My standard delivery has taken 10 business days and has not arrived.",
    "I am a premium customer. Is chat support available at midnight?",
    "Can I exchange my purchase for a different color?"
]

for question in test_questions:
    print(f"\nCUSTOMER: {question}")
    result = customer_support_assistant(question)
    print(result.model_dump_json(indent=2))
    print("=" * 100)

# Optional Challenge — Turn the Project into a Simple Interactive App

Use this loop to interact with the support assistant directly inside Colab.


In [ ]:
print("AI Customer Support Assistant")
print("Type 'exit' to stop.\n")

while True:
    question = input("Customer: ").strip()

    if question.lower() == "exit":
        print("Session ended.")
        break

    result = customer_support_assistant(question)

    print("\nAnswer:", result.answer)
    print("Category:", result.category)
    print("Priority:", result.priority)
    print("Human agent required:", result.needs_human_agent)
    print("Sources:", ", ".join(result.source_policies))
    print("-" * 80)

# Final Hands-On Assignment

Build a **Course Recommendation Assistant** using LangChain and Gemini.

## Requirements

Create at least five sample course documents. Each course should include:

- course name;
- skills taught;
- experience level;
- duration;
- prerequisites.

Then build a RAG application that accepts questions such as:

> I am an SAP ABAP developer with no AI experience. Which course should I take first to learn SAP Business AI?

Your system should return structured output with:

- `recommended_courses`
- `reason`
- `prerequisites`
- `learning_sequence`
- `confidence`

## Bonus requirements

1. Add a custom tool that calculates total learning hours.
2. Add conversation history.
3. Return the result as a Pydantic model.
4. Add source metadata to each recommendation.
5. Build a Streamlit front end.


# Common Errors and Troubleshooting

## 1. Authentication error

Check that `GOOGLE_API_KEY` is correctly configured.

```python
import os
print(bool(os.getenv("GOOGLE_API_KEY")))
```

Never print the actual key.

## 2. Model not found

Gemini model availability can change. Check the current Google Gemini model documentation and replace the model ID in the initialization cell.

## 3. Import error

Re-run the installation cell and restart the Colab runtime when necessary.

## 4. Rate-limit or quota error

Wait for quota availability, reduce repeated calls, or review the quota associated with the API key/project.

## 5. Poor RAG answers

Inspect retrieval independently:

```python
retriever.invoke("your question")
```

Then improve document quality, chunk sizes, metadata, the number of retrieved chunks, or the answer prompt.


# Key Takeaways

You have now practiced the main building blocks of a LangChain application:

**Prompt → Model → Parser → History → Tools → Embeddings → Retriever → RAG → Structured Application Output**

A sensible next progression is:

1. Add real PDF or web documents.
2. Add a persistent vector database.
3. Add LangSmith tracing and evaluation.
4. Add LangGraph for stateful multi-step agent workflows.
5. Build a Streamlit front end.
6. Deploy the application to a cloud platform.

---

## Official documentation used for this notebook

- LangChain Google Gemini chat-model integration
- LangChain Google Gemini embeddings integration
- LangChain structured-output documentation
- Google Gemini API model documentation

The notebook intentionally uses the current `langchain-google-genai` integration instead of older deprecated examples.
